# RAG from scratch

**RAG = Retrieval-Augmented Generation.** Instead of asking a language model to answer from memory, we first *retrieve* the most relevant text from our own documents, then hand it to the model together with the question, so the answer is *grounded* in real facts.

The pipeline, built one step at a time in this notebook:

```
documents -> chunks -> embeddings -> (question -> embedding) -> similarity search -> augmented prompt -> answer
```

Use case: a small retail customer-support assistant (returns, shipping, warranty, payment, opening hours).
Run each cell in order and read the output before moving on.

## Step 0 · Setup
Load the API key from `.env` and create the client. The small helper functions live in `rag_utils.py`, and they have tests (`pytest`).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai

from rag_utils import build_prompt, chunk_text, cosine_similarity, embed_texts, top_k

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
GEN_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.7-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2")
print("Client ready. Models:", GEN_MODEL, "|", EMBED_MODEL)

## Step 1 · Load the documents
One text file per topic in the `docs/` folder.

In [ ]:
documents = {p.name: p.read_text(encoding="utf-8") for p in sorted(Path("docs").glob("*.txt"))}
for name, text in documents.items():
    print(f"{name}: {len(text)} characters")

## Step 2 · Chunking
Models work better with small, focused pieces of text. We split each document into paragraph-level chunks and remember where each chunk came from.

In [ ]:
chunks, sources = [], []
for name, text in documents.items():
    for chunk in chunk_text(text, max_chars=500):
        chunks.append(chunk)
        sources.append(name)

print(len(chunks), "chunks")
print("Example chunk:", chunks[0])

## Step 3 · Embeddings
An *embedding* turns text into a list of numbers (a vector) so that texts with similar meaning end up close together. We embed every chunk once.

In [ ]:
chunk_vectors = embed_texts(client, EMBED_MODEL, chunks)
print("Vectors:", len(chunk_vectors), "| dimensions per vector:", len(chunk_vectors[0]))

## Step 4 · Ask a question and embed it
The question goes through the same embedding model, so it lives in the same vector space as the chunks.

In [ ]:
question = "How long do I have to return an item?"
question_vector = embed_texts(client, EMBED_MODEL, [question])[0]
print("Question vector has", len(question_vector), "dimensions")

## Step 5 · Retrieve the best chunks
Cosine similarity measures how close two vectors point (1 = same meaning direction, 0 = unrelated). We keep the top 2 chunks.

In [ ]:
hits = top_k(question_vector, chunk_vectors, k=2)
for index, score in hits:
    print(f"score {score:.3f}  from {sources[index]}")
    print("   ", chunks[index][:120].replace("\n", " "), "...")

## Step 6 · Augment the prompt
This is the *A* in RAG: we put the retrieved text and the question into one prompt and tell the model to use only that context.

In [ ]:
context_chunks = [chunks[i] for i, _ in hits]
prompt = build_prompt(question, context_chunks)
print(prompt)

## Step 7 · Generate the answer
Now the model answers, grounded in our documents.

In [ ]:
response = client.models.generate_content(model=GEN_MODEL, contents=prompt)
print(response.text)

## Step 8 · Try it yourself

1. Change `question` in Step 4 (for example *"Is delivery free?"* or *"Does the warranty cover water damage?"*) and run Steps 4 to 7 again.
2. Ask something that is **not** in the documents (for example *"Do you sell shoes?"*). A well-built RAG system should say it doesn't know.
3. Change `max_chars` in Step 2 and `k` in Step 5 and see how the answers change.

## What I learned

- Retrieval quality decides answer quality: a wrong chunk gives a confident wrong answer.
- Chunk size is a trade-off between too little context and too much noise.
- The instruction *"use ONLY the context"* is what keeps the model honest.

## Next

Self-healing RAG: check whether the retrieved context really answers the question, rewrite the query and retry when it does not.